### **What's next?**

Goals:
- import scikit-learn library to split dataset into training and test data
- encoding categorical columns using one-hot encoding (ML only understand numerical values)
- select an ml model that aligns with the data
- perform analysis to ensure that the model is capable of predicting

Error:
- the model couldn't train after encoding since this is a big dataset with more than 400k entires.

Solution:
- we can minimize the number of entires in the dataset into 50,000 or below that for better performance and less space consumption

In [1]:
import pandas as pd

df = pd.read_csv("/home/nabusa/Documents/python_essentials/dataset/car_prices.csv")

In [2]:
# Removing unecessary columns - vin, seller, saledate
# df = df.drop('saledate', axis = 1)

# looking for null values again.
df = df.dropna()
df.dtypes

year              int64
make                str
model               str
trim                str
body                str
transmission        str
vin                 str
state               str
condition       float64
odometer        float64
color               str
interior            str
seller              str
mmr             float64
sellingprice    float64
saledate            str
dtype: object

In [3]:
df.shape
df.isnull().sum()

# Missing values were present in several features, including make, model, trim, and body. 
# Since the dataset is large, rows containing null values were removed using dropna() to simplify preprocessing 
# and ensure that the machine learning model is trained on complete records only.

year            0
make            0
model           0
trim            0
body            0
transmission    0
vin             0
state           0
condition       0
odometer        0
color           0
interior        0
seller          0
mmr             0
sellingprice    0
saledate        0
dtype: int64

### **Feature Engineering**

this is the crucial phase of the ml building lifecyle. in this phase, we can add features (new important columns) or improve the features which will be easier for the ml model to understand.

- Goal: create ``car_age``
- Reason: for ml to understand, how old the vehicle is since manufactured.
- solution: we will create a column, by subracting the manufacturing_year of the car from the present year 

In [4]:
df['car_age'] = 2026 - df['year']
df

#insight:
#this column is created to learn that as the car ages, its selling price also decreases

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate,car_age
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),11
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST),11
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST),12
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST),11
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST),12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558831,2011,BMW,5 Series,528i,Sedan,automatic,wbafr1c53bc744672,fl,39.0,66403.0,white,brown,lauderdale imports ltd bmw pembrok pines,20300.0,22800.0,Tue Jul 07 2015 06:15:00 GMT-0700 (PDT),15
558833,2012,Ram,2500,Power Wagon,Crew Cab,automatic,3c6td5et6cg112407,wa,5.0,54393.0,white,black,i -5 uhlmann rv,30200.0,30800.0,Wed Jul 08 2015 09:30:00 GMT-0700 (PDT),14
558834,2012,BMW,X5,xDrive35d,SUV,automatic,5uxzw0c58cl668465,ca,48.0,50561.0,black,black,financial services remarketing (lease),29800.0,34000.0,Wed Jul 08 2015 09:30:00 GMT-0700 (PDT),14
558835,2015,Nissan,Altima,2.5 S,sedan,automatic,1n4al3ap0fc216050,ga,38.0,16658.0,white,black,enterprise vehicle exchange / tra / rental / t...,15100.0,11100.0,Thu Jul 09 2015 06:45:00 GMT-0700 (PDT),11


### **Creating a Sample Dataset**

Goal: Reduce the dataset size to lower memory usage and improve training efficiency.

Task:
- reduce the dataset size to 50000
- preserve reproducibility using `random_state = 42`

In [5]:
df_sample = df.sample(n = 50000, random_state= 42)

### **Selecting the important features**

Goal: Choosing the most influential variables identified during EDA and separate them into input features and the target variable for machine learning.
`make, body, condition, odometer, mmr, car_age, transmission, year`

splitting the target feature from the input feature is same but we're using only the limited size of the dataset

In [6]:
feature_selection = [
    'make',
    'body',
    'condition',
    'odometer',
    'mmr',
    'car_age',
    'transmission',
    'year'
]

X = df_sample[feature_selection]
y = df_sample['sellingprice']

### **Identifying Categorical and Numerical columns**

Before encoding, we need to determine:
- Which columns are numeric
- Which columns are categorical (text)

This helps us:
- know what must be encoded,
- and understand the structure of the feature set.

use case: `X.select_datatypes[].columns` - identifies the set of columns that you need

In [7]:
#df = df.drop(columns=['seller'])
df.dtypes

year              int64
make                str
model               str
trim                str
body                str
transmission        str
vin                 str
state               str
condition       float64
odometer        float64
color               str
interior            str
seller              str
mmr             float64
sellingprice    float64
saledate            str
car_age           int64
dtype: object

In [8]:
cat_col = X.select_dtypes(include= 'str').columns
num_col = X.select_dtypes(include = ['int64', 'float64']).columns

print(f"\nCategorical columns: {cat_col}") #there are 8 columns which are categorical
print(f"\nNumerical columns: {num_col}") # there are 6 columns which are categorical

#insights:
# The selected features were classified into numerical and categorical variables. 
# This distinction is necessary because categorical features must be transformed into numerical format 
# before training the machine learning model.


Categorical columns: Index(['make', 'body', 'transmission'], dtype='str')

Numerical columns: Index(['condition', 'odometer', 'mmr', 'car_age', 'year'], dtype='str')


### **One-hot Encoding**

ML cannot understand texts. it needs to be converted into numbers

we will use: `pd.get_dummies(input feature, columns=, drop_first = True)`

when you use drop_first, it deletes the specified column/s inside the input features dataset and creates a separate encoded columns, to avoid duplication of the category columns

In [9]:
X_encoded = pd.get_dummies(X, columns = cat_col, drop_first=True)

print(f"X encoded shape: {X_encoded.shape}")
print(f'\nX encoded head: {X_encoded.head()}')

#insight:
# Categorical variables were transformed into numerical format using one-hot encoding with pd.get_dummies(). 
# The drop_first=True parameter was used to remove redundant categories and reduce multicollinearity.

X encoded shape: (50000, 129)

X encoded head:         condition  odometer      mmr  car_age  year  make_Aston Martin  \
297590       34.0  121222.0   4150.0       20  2006              False   
90053        46.0   42814.0  20400.0       16  2010              False   
341051       42.0   20032.0  22900.0       12  2014              False   
362239       23.0   52864.0  12050.0       13  2013              False   
480321       37.0   72306.0   8725.0       14  2012              False   

        make_Audi  make_BMW  make_Bentley  make_Buick  ...  body_regular-cab  \
297590      False     False         False       False  ...             False   
90053       False     False         False       False  ...             False   
341051      False     False         False       False  ...             False   
362239      False     False         False       False  ...             False   
480321      False     False         False       False  ...             False   

        body_sedan  body_su

### **Train-Test split**

This is where we divide the dataset into two parts:
- Training Set → used to teach the model.
- Testing Set → used to evaluate how well the model performs on unseen data.

Reason: if we train and test on the same data, the model may simply memorize the answers. To measure real performance, we reserve a portion of the data that the model has never seen before.

use case: `inputfeat_train, inputfeat_test, targetfeat_train, targetfeat_test = train_test_split(categorical column encoded, target column, test_size to determine how much data to be cosidered for testing, random_state)`



In [10]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [11]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

#insight:
# The dataset was divided into training and testing sets using an 80:20 ratio (80% training, 20% testing). 
# The training set was used to fit the machine learning model,
# while the testing set was reserved for evaluating its predictive performance on unseen data.

(40000, 129)
(10000, 129)
(40000,)
(10000,)


### **Training Machine Learning Model**

Here, we will the train the model by building an algorithm, where it finds the relationship between input features(X_train) and target feature(y_train)

use of ML model - LinearRegression, which is finding the mathematical equation. this is useful for model to predict the output.

Training the model is the process of teaching the algorithm to recognize patterns in historical car data so it can predict the selling price of new cars accurately.

In [12]:
#importing the model
from sklearn.linear_model import LinearRegression

# creating a model
lr_model = LinearRegression()

#training the model
lr_model.fit(X_train, y_train)


#insights:
# A Linear Regression model was initialized and trained using the training dataset. 
# During this process, the algorithm learned the mathematical relationship between the selected vehicle features 
# and their corresponding selling prices.

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


### **Generating predictions (y_pred)**

Goal: we're using the trained model to predict the prices for the test data

`(X_test)` is our unseen data, `(y_pred)` is where the predicted data will be stored, preview the first few predicted data

In [13]:
y_predict = lr_model.predict(X_test)

print(y_predict[:10])

#insights:
# The trained Linear Regression model was used to predict selling prices on the test dataset. 
# These predictions will be compared with the actual values in y_test to evaluate model performance.

[17755.40053097  6444.69220779 15726.63088775 17134.99832766
 12555.87929624 27174.55375456 20218.58210443  9004.7878568
  6828.19263684  6102.94420532]


### **Evaluating the Model**

Goal: To measure how accurately the Linear Regression model predicts used car selling prices by comparing the predicted values with the actual values.

Reason:
- to predict how far or close the predicted values are from the actual selling prices
- check whether it is reliable or not 

Use Cases:
- MAE - Mean Absolute Error: avg absolute diff between `actual` and `predicted` prices
- RMSE - Root Mean Squared Error: measure prediction error, while penalising larger mistakes more heavily.
- R-square: measures the variation in selling prices, explained by the model

In [14]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import numpy

# Evaluating the metrics. 
mae = mean_absolute_error(y_test, y_predict)
rmse = root_mean_squared_error(y_test, y_predict)
r2 = r2_score(y_test, y_predict)

print(f"\nMean Absolute Error: {round(mae, 2)}")
print(f"Root Mean Squared Error: {round(rmse, 2)}")
print(f"R2 Score: {round(r2, 2)}")


Mean Absolute Error: 1042.71
Root Mean Squared Error: 1623.06
R2 Score: 0.97


### **Conclusion**



A Linear Regression model was trained to predict used car selling prices using *selected vehicle* characteristics and *market valuation* features. The model achieved a *Mean Absolute Error* of approximately `1,043`, a *Root Mean Squared Error* of `1,623`, and an *R² score* of `97.04%`, indicating that it explains nearly all of the variation in selling prices. These results demonstrate that the chosen features, particularly the MMR market value, are highly effective predictors of vehicle resale prices.

In [16]:
import joblib

joblib.dump(lr_model, "car_price_model.pkl")
joblib.dump(X_encoded.columns, "model_columns.pkl")

['model_columns.pkl']